### Configuration


In [ ]:
#Prompts
"""
Centralized prompt templates for all agents in the research system.
"""

PLANNER_PROMPT = """
You are a senior business intelligence research strategist.

Given a research topic, generate sub-questions across ALL of these categories:

- definitional: What is X? How is X defined or measured in the industry?
- causal: Why does X happen? What are the root drivers of X?
- comparative: How does X differ across competitors, time periods, or markets?
- quantitative: What are the key metrics, figures, or trends for X?
- contrarian: What challenges the mainstream view of X?
- procedural: How is X implemented, evaluated, or managed?
- gap_seeking: What is unknown, contested, or under-researched about X?

Topic: {topic}

Rules:
- Generate 2-3 questions per category (14-21 questions total)
- Make each question specific and answerable from documents
- Avoid vague or overlapping questions

Return ONLY valid JSON, no preamble:
[
  {{"id": "q1", "type": "definitional", "question": "...", "status": "pending"}}
]
"""

SYNTHESIZER_PROMPT = """
You are a business intelligence analyst writing a structured research report.

STRICT RULES:
1. Use ONLY the provided context. Never add external knowledge.
2. Every factual claim must reference a source chunk by its [chunk_id].
3. If a sub-question cannot be answered from context, explicitly state:
   'Insufficient data: <question_text>'
4. Output format must be exactly as specified below.

Sub-questions to address: {sub_questions}

Context chunks:
{context}

Output format (JSON):
{{
  "report": {{
    "title": "Research Report: <topic>",
    "sections": [
      {{
        "heading": "<section_name>",
        "content": "<grounded_text_with_[chunk_id]_citations>",
        "claims": [
          {{
            "claim": "<specific_claim>",
            "source_chunks": ["chunk_id_1", "chunk_id_2"]
          }}
        ]
      }}
    ],
    "unanswered_questions": ["<question_text>"],
    "confidence_score": 0.85
  }}
}}
"""

GAP_ANALYSIS_PROMPT = """
You are a research quality auditor.

Review the synthesis report and determine if there are significant gaps or weaknesses.

Report: {report}
Original sub-questions: {sub_questions}
Retrieval scores: {retrieval_scores}

Evaluate:
1. Coverage: Are all sub-questions adequately addressed?
2. Confidence: Are confidence scores above {min_confidence}?
3. Citations: Are claims properly grounded in sources?
4. Gaps: Are there unanswered questions or weak sections?

Return JSON:
{{
  "has_gaps": true/false,
  "gap_details": [
    {{
      "type": "coverage|confidence|citation",
      "description": "...",
      "affected_questions": ["q1", "q2"]
    }}
  ],
  "recommendation": "continue|refine_questions|finalize"
}}
"""

HYDE_PROMPT = """
Given a research question, generate a hypothetical ideal answer that would perfectly address this question.
This will be used to improve retrieval by finding documents similar to this ideal answer.

Question: {question}

Generate a detailed, specific hypothetical answer (2-3 paragraphs) that:
- Directly addresses the question
- Includes specific terminology and concepts likely to appear in relevant documents
- Maintains a professional, analytical tone

Hypothetical Answer:
"""

RERANKING_PROMPT = """
Given a query and a document chunk, rate the relevance of the chunk to the query on a scale of 0-1.

Query: {query}
Chunk: {chunk}

Consider:
- Direct relevance to the query topic
- Presence of key terms and concepts
- Quality and specificity of information

Return only a number between 0 and 1:
"""


In [ ]:
# Settings
"""
Global configuration settings for the Autonomous Research Analyst.
Loads environment variables and provides centralized access to configuration.
"""

import os
from pathlib import Path
from typing import List
from dotenv import load_dotenv
from loguru import logger

# Load environment variables
load_dotenv()

# Base paths
BASE_DIR = Path(__file__).parent.parent
DATA_DIR = BASE_DIR / "data"
UPLOADS_DIR = DATA_DIR / "uploads"
CHROMA_DIR = DATA_DIR / "chroma_db"
OUTPUT_DIR = BASE_DIR / os.getenv("OUTPUT_DIR", "outputs")
LOGS_DIR = BASE_DIR / "logs"

# Ensure directories exist
for directory in [DATA_DIR, UPLOADS_DIR, CHROMA_DIR, OUTPUT_DIR, LOGS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Ollama Configuration
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3")
OLLAMA_EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")

# ChromaDB Configuration
CHROMA_PERSIST_DIR = str(CHROMA_DIR)
CHROMA_COLLECTION_NAME = os.getenv("CHROMA_COLLECTION_NAME", "research_docs")

# Retrieval Configuration
RETRIEVAL_TOP_K = int(os.getenv("RETRIEVAL_TOP_K", "10"))
RERANK_TOP_K = int(os.getenv("RERANK_TOP_K", "5"))
HYBRID_ALPHA = float(os.getenv("HYBRID_ALPHA", "0.5"))

# Agent Configuration
MAX_RESEARCH_ITERATIONS = int(os.getenv("MAX_RESEARCH_ITERATIONS", "3"))
MIN_CONFIDENCE_SCORE = float(os.getenv("MIN_CONFIDENCE_SCORE", "0.7"))

# Slack Configuration
SLACK_BOT_TOKEN = os.getenv("SLACK_BOT_TOKEN", "")
SLACK_APP_TOKEN = os.getenv("SLACK_APP_TOKEN", "")
SLACK_SIGNING_SECRET = os.getenv("SLACK_SIGNING_SECRET", "")

# Teams Configuration
TEAMS_APP_ID = os.getenv("TEAMS_APP_ID", "")
TEAMS_APP_PASSWORD = os.getenv("TEAMS_APP_PASSWORD", "")

# Logging Configuration
LOG_LEVEL = os.getenv("LOG_LEVEL", "INFO")
LOG_FILE = LOGS_DIR / "research_analyst.log"

# Output Configuration
REPORT_FORMATS: List[str] = os.getenv("REPORT_FORMATS", "pdf,markdown,json").split(",")

# Chunking Configuration
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 200

# Configure logger
logger.add(
    LOG_FILE,
    rotation="10 MB",
    retention="7 days",
    level=LOG_LEVEL,
    format="{time:YYYY-MM-DD HH:mm:ss} | {level} | {name}:{function}:{line} | {message}"
)

logger.info("Configuration loaded successfully")


### Ingestion

In [ ]:
# Agentic Chunker
from langchain_core.prompts import ChatPromptTemplate
import uuid
from langchain_community.llms import Ollama
import os
from typing import Optional
try:
    from langchain_core.pydantic_v1 import BaseModel
except ImportError:
    from pydantic import BaseModel
from langchain.chains import create_extraction_chain_pydantic
from dotenv import load_dotenv
from rich import print

load_dotenv()

class AgenticChunker:
    def __init__(self, model_name='mistral', base_url='http://localhost:11434'):
        """
        Initialize AgenticChunker with free local Ollama LLM
        
        Args:
            model_name: Name of the Ollama model to use (default: 'mistral')
            base_url: Ollama server URL (default: 'http://localhost:11434')
        """
        self.chunks = {}
        self.id_truncate_limit = 5

        # Whether or not to update/refine summaries and titles as you get new information
        self.generate_new_metadata_ind = True
        self.print_logging = True

        # Use free local Ollama instead of paid OpenAI API
        self.llm = Ollama(
            model=model_name,
            base_url=base_url,
            temperature=0.25,
            top_p=0.9
        )
        

    def add_propositions(self, propositions):
        for proposition in propositions:
            self.add_proposition(proposition)
    
    def add_proposition(self, proposition):
        if self.print_logging:
            print (f"\nAdding: '{proposition}'")

        # If it's your first chunk, just make a new chunk and don't check for others
        if len(self.chunks) == 0:
            if self.print_logging:
                print ("No chunks, creating a new one")
            self._create_new_chunk(proposition)
            return

        chunk_id = self._find_relevant_chunk(proposition)

        # If a chunk was found then add the proposition to it
        if chunk_id:
            if self.print_logging:
                print (f"Chunk Found ({self.chunks[chunk_id]['chunk_id']}), adding to: {self.chunks[chunk_id]['title']}")
            self.add_proposition_to_chunk(chunk_id, proposition)
            return
        else:
            if self.print_logging:
                print ("No chunks found")
            # If a chunk wasn't found, then create a new one
            self._create_new_chunk(proposition)
        

    def add_proposition_to_chunk(self, chunk_id, proposition):
        # Add then
        self.chunks[chunk_id]['propositions'].append(proposition)

        # Then grab a new summary
        if self.generate_new_metadata_ind:
            self.chunks[chunk_id]['summary'] = self._update_chunk_summary(self.chunks[chunk_id])
            self.chunks[chunk_id]['title'] = self._update_chunk_title(self.chunks[chunk_id])

    def _update_chunk_summary(self, chunk):
        """
        If you add a new proposition to a chunk, you may want to update the summary or else they could get stale
        """
        PROMPT = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are the steward of a group of chunks which represent groups of sentences that talk about a similar topic
                    A new proposition was just added to one of your chunks, you should generate a very brief 1-sentence summary which will inform viewers what a chunk group is about.

                    A good summary will say what the chunk is about, and give any clarifying instructions on what to add to the chunk.

                    You will be given a group of propositions which are in the chunk and the chunks current summary.

                    Your summaries should anticipate generalization. If you get a proposition about apples, generalize it to food.
                    Or month, generalize it to "date and times".

                    Example:
                    Input: Proposition: Greg likes to eat pizza
                    Output: This chunk contains information about the types of food Greg likes to eat.

                    Only respond with the chunk new summary, nothing else.
                    """,
                ),
                ("user", "Chunk's propositions:\n{proposition}\n\nCurrent chunk summary:\n{current_summary}"),
            ]
        )

        runnable = PROMPT | self.llm

        new_chunk_summary = runnable.invoke({
            "proposition": "\n".join(chunk['propositions']),
            "current_summary" : chunk['summary']
        })

        return new_chunk_summary
    
    def _update_chunk_title(self, chunk):
        """
        If you add a new proposition to a chunk, you may want to update the title or else it can get stale
        """
        PROMPT = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are the steward of a group of chunks which represent groups of sentences that talk about a similar topic
                    A new proposition was just added to one of your chunks, you should generate a very brief updated chunk title which will inform viewers what a chunk group is about.

                    A good title will say what the chunk is about.

                    You will be given a group of propositions which are in the chunk, chunk summary and the chunk title.

                    Your title should anticipate generalization. If you get a proposition about apples, generalize it to food.
                    Or month, generalize it to "date and times".

                    Example:
                    Input: Summary: This chunk is about dates and times that the author talks about
                    Output: Date & Times

                    Only respond with the new chunk title, nothing else.
                    """,
                ),
                ("user", "Chunk's propositions:\n{proposition}\n\nChunk summary:\n{current_summary}\n\nCurrent chunk title:\n{current_title}"),
            ]
        )

        runnable = PROMPT | self.llm

        updated_chunk_title = runnable.invoke({
            "proposition": "\n".join(chunk['propositions']),
            "current_summary" : chunk['summary'],
            "current_title" : chunk['title']
        })

        return updated_chunk_title

    def _get_new_chunk_summary(self, proposition):
        PROMPT = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are the steward of a group of chunks which represent groups of sentences that talk about a similar topic
                    You should generate a very brief 1-sentence summary which will inform viewers what a chunk group is about.

                    A good summary will say what the chunk is about, and give any clarifying instructions on what to add to the chunk.

                    You will be given a proposition which will go into a new chunk. This new chunk needs a summary.

                    Your summaries should anticipate generalization. If you get a proposition about apples, generalize it to food.
                    Or month, generalize it to "date and times".

                    Example:
                    Input: Proposition: Greg likes to eat pizza
                    Output: This chunk contains information about the types of food Greg likes to eat.

                    Only respond with the new chunk summary, nothing else.
                    """,
                ),
                ("user", "Determine the summary of the new chunk that this proposition will go into:\n{proposition}"),
            ]
        )

        runnable = PROMPT | self.llm

        new_chunk_summary = runnable.invoke({
            "proposition": proposition
        })

        return new_chunk_summary
    
    def _get_new_chunk_title(self, summary):
        PROMPT = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    You are the steward of a group of chunks which represent groups of sentences that talk about a similar topic
                    You should generate a very brief few word chunk title which will inform viewers what a chunk group is about.

                    A good chunk title is brief but encompasses what the chunk is about

                    You will be given a summary of a chunk which needs a title

                    Your titles should anticipate generalization. If you get a proposition about apples, generalize it to food.
                    Or month, generalize it to "date and times".

                    Example:
                    Input: Summary: This chunk is about dates and times that the author talks about
                    Output: Date & Times

                    Only respond with the new chunk title, nothing else.
                    """,
                ),
                ("user", "Determine the title of the chunk that this summary belongs to:\n{summary}"),
            ]
        )

        runnable = PROMPT | self.llm

        new_chunk_title = runnable.invoke({
            "summary": summary
        })

        return new_chunk_title


    def _create_new_chunk(self, proposition):
        new_chunk_id = str(uuid.uuid4())[:self.id_truncate_limit] # I don't want long ids
        new_chunk_summary = self._get_new_chunk_summary(proposition)
        new_chunk_title = self._get_new_chunk_title(new_chunk_summary)

        self.chunks[new_chunk_id] = {
            'chunk_id' : new_chunk_id,
            'propositions': [proposition],
            'title' : new_chunk_title,
            'summary': new_chunk_summary,
            'chunk_index' : len(self.chunks)
        }
        if self.print_logging:
            print (f"Created new chunk ({new_chunk_id}): {new_chunk_title}")
    
    def get_chunk_outline(self):
        """
        Get a string which represents the chunks you currently have.
        This will be empty when you first start off
        """
        chunk_outline = ""

        for chunk_id, chunk in self.chunks.items():
            single_chunk_string = f"""Chunk ({chunk['chunk_id']}): {chunk['title']}\nSummary: {chunk['summary']}\n\n"""
        
            chunk_outline += single_chunk_string
        
        return chunk_outline

    def _find_relevant_chunk(self, proposition):
        current_chunk_outline = self.get_chunk_outline()

        PROMPT = ChatPromptTemplate.from_messages(
            [
                (
                    "system",
                    """
                    Determine whether or not the "Proposition" should belong to any of the existing chunks.

                    A proposition should belong to a chunk of their meaning, direction, or intention are similar.
                    The goal is to group similar propositions and chunks.

                    If you think a proposition should be joined with a chunk, return the chunk id.
                    If you do not think an item should be joined with an existing chunk, just return "No chunks"

                    Example:
                    Input:
                        - Proposition: "Greg really likes hamburgers"
                        - Current Chunks:
                            - Chunk ID: 2n4l3d
                            - Chunk Name: Places in San Francisco
                            - Chunk Summary: Overview of the things to do with San Francisco Places

                            - Chunk ID: 93833k
                            - Chunk Name: Food Greg likes
                            - Chunk Summary: Lists of the food and dishes that Greg likes
                    Output: 93833k
                    """,
                ),
                ("user", "Current Chunks:\n--Start of current chunks--\n{current_chunk_outline}\n--End of current chunks--"),
                ("user", "Determine if the following statement should belong to one of the chunks outlined:\n{proposition}"),
            ]
        )

        runnable = PROMPT | self.llm

        chunk_found = runnable.invoke({
            "proposition": proposition,
            "current_chunk_outline": current_chunk_outline
        })

        # Pydantic data class
        class ChunkID(BaseModel):
            """Extracting the chunk id"""
            chunk_id: Optional[str]
            
        # Extraction to catch-all LLM responses. This is a bandaid
        extraction_chain = create_extraction_chain_pydantic(pydantic_schema=ChunkID, llm=self.llm)
        extraction_found = extraction_chain.invoke(chunk_found)["text"]
        if extraction_found:
            chunk_found = extraction_found[0].chunk_id

        # If you got a response that isn't the chunk id limit, chances are it's a bad response or it found nothing
        # So return nothing
        if len(chunk_found) != self.id_truncate_limit:
            return None

        return chunk_found
    
    def get_chunks(self, get_type='dict'):
        """
        This function returns the chunks in the format specified by the 'get_type' parameter.
        If 'get_type' is 'dict', it returns the chunks as a dictionary.
        If 'get_type' is 'list_of_strings', it returns the chunks as a list of strings, where each string is a proposition in the chunk.
        """
        if get_type == 'dict':
            return self.chunks
        if get_type == 'list_of_strings':
            chunks = []
            for chunk_id, chunk in self.chunks.items():
                chunks.append(" ".join([x for x in chunk['propositions']]))
            return chunks
    
    def pretty_print_chunks(self):
        print (f"\nYou have {len(self.chunks)} chunks\n")
        for chunk_id, chunk in self.chunks.items():
            print(f"Chunk #{chunk['chunk_index']}")
            print(f"Chunk ID: {chunk_id}")
            print(f"Summary: {chunk['summary']}")
            print(f"Propositions:")
            for prop in chunk['propositions']:
                print(f"    -{prop}")
            print("\n\n")

    def pretty_print_chunk_outline(self):
        print ("Chunk Outline\n")
        print(self.get_chunk_outline())

if __name__ == "__main__":
    ac = AgenticChunker()

    ## Comment and uncomment the propositions to your hearts content
    propositions = [
        'The month is October.',
        'The year is 2023.',
        "One of the most important things that I didn't understand about the world as a child was the degree to which the returns for performance are superlinear.",
        'Teachers and coaches implicitly told us that the returns were linear.',
        "I heard a thousand times that 'You get out what you put in.'",
        'Teachers and coaches meant well.',
        "The statement that 'You get out what you put in' is rarely true.",
        "If your product is only half as good as your competitor's product, you do not get half as many customers.",
        "You get no customers if your product is only half as good as your competitor's product.",
        'You go out of business if you get no customers.',
        'The returns for performance are superlinear in business.',
        'Some people think the superlinear returns for performance are a flaw of capitalism.',
        'Some people think that changing the rules of capitalism would stop the superlinear returns for performance from being true.',
        'Superlinear returns for performance are a feature of the world.',
        'Superlinear returns for performance are not an artifact of rules that humans have invented.',
        'The same pattern of superlinear returns is observed in fame.',
        'The same pattern of superlinear returns is observed in power.',
        'The same pattern of superlinear returns is observed in military victories.',
        'The same pattern of superlinear returns is observed in knowledge.',
        'The same pattern of superlinear returns is observed in benefit to humanity.',
        'In fame, power, military victories, knowledge, and benefit to humanity, the rich get richer.'
    ]
    
    ac.add_propositions(propositions)
    ac.pretty_print_chunks()
    ac.pretty_print_chunk_outline()
    print (ac.get_chunks(get_type='list_of_strings'))

In [ ]:
#Chunker
"""
Proposition-Based Chunking using Agentic Chunker with FREE online HuggingFace LLMs

This module extracts propositions from text and uses the AgenticChunker
to create semantically meaningful chunks.
"""

from langchain_core.prompts import ChatPromptTemplate
from langchain_community.llms import Ollama
try:
    from langchain_core.pydantic_v1 import BaseModel
except ImportError:
    from pydantic import BaseModel
from langchain.chains import create_extraction_chain_pydantic
from typing import List, Optional
from ingest.agentic_chunker import AgenticChunker
from rich import print


class Sentences(BaseModel):
    """Pydantic model for extracting sentences/propositions"""
    sentences: List[str]


class PropositionExtractor:
    """Extract propositions from text using local Ollama LLM"""
    
    def __init__(self, model_name='mistral', base_url='http://localhost:11434'):
        """
        Initialize proposition extractor with free local Ollama
        
        Args:
            model_name: Ollama model to use (default: 'mistral')
            base_url: Ollama server URL (default: 'http://localhost:11434')
        """
        # Use free local Ollama
        self.llm = Ollama(
            model=model_name,
            base_url=base_url,
            temperature=0.25,
            top_p=0.9
        )
        
        
        # Prompt for extracting propositions
        self.prompt = ChatPromptTemplate.from_messages([
            (
                "system",
                """You are an expert at breaking down text into atomic propositions.
                
                A proposition is a single, self-contained statement that conveys one fact or idea.
                
                Your task:
                1. Read the input text carefully
                2. Break it down into individual propositions
                3. Each proposition should be a complete sentence
                4. Maintain the original meaning and context
                5. Return a list of propositions
                
                Example:
                Input: "John went to the store and bought apples. He paid $5."
                Output: 
                - "John went to the store"
                - "John bought apples"
                - "He paid $5"
                
                Return only the propositions as a numbered list, nothing else.
                """
            ),
            ("user", "Extract propositions from this text:\n\n{input}")
        ])
        
        self.runnable = self.prompt | self.llm
        self.extraction_chain = create_extraction_chain_pydantic(
            pydantic_schema=Sentences, 
            llm=self.llm
        )
    
    def get_propositions(self, text: str) -> List[str]:
        """
        Extract propositions from a text paragraph
        
        Args:
            text: Input text to extract propositions from
            
        Returns:
            List of proposition strings
        """
        if not text or not text.strip():
            return []
        
        try:
            # Get LLM output
            runnable_output = self.runnable.invoke({"input": text})
            
            # Extract structured propositions
            extracted = self.extraction_chain.invoke(runnable_output)
            
            if extracted and "text" in extracted and len(extracted["text"]) > 0:
                propositions = extracted["text"][0].sentences
                return propositions
            
            # Fallback: parse the output manually if extraction fails
            lines = runnable_output.strip().split('\n')
            propositions = []
            for line in lines:
                # Remove numbering and clean up
                cleaned = line.strip()
                if cleaned and len(cleaned) > 10:  # Filter out very short lines
                    # Remove common prefixes like "1.", "-", "*"
                    for prefix in ['- ', '* ', '• ']:
                        if cleaned.startswith(prefix):
                            cleaned = cleaned[len(prefix):].strip()
                    # Remove numbered prefixes like "1. ", "2. "
                    if cleaned[0].isdigit() and '. ' in cleaned[:4]:
                        cleaned = cleaned.split('. ', 1)[1].strip()
                    
                    if cleaned:
                        propositions.append(cleaned)
            
            return propositions
            
        except Exception as e:
            print(f"[red]Error extracting propositions: {e}[/red]")
            # Fallback: split by sentences
            return [s.strip() + '.' for s in text.split('.') if s.strip()]


def chunk_text_with_propositions(
    text: str,
    model_name: str = 'mistralai/Mistral-7B-Instruct-v0.2',
    max_paragraphs: Optional[int] = None
) -> List[str]:
    """
    Main function to chunk text using proposition-based agentic chunking
    
    Args:
        text: Input text to chunk
        model_name: HuggingFace model name (default: 'mistralai/Mistral-7B-Instruct-v0.2')
        max_paragraphs: Maximum number of paragraphs to process (None = all)
        
    Returns:
        List of semantically meaningful text chunks
    """
    print("[bold blue]#### Proposition-Based Chunking ####[/bold blue]")
    
    # Initialize extractors
    extractor = PropositionExtractor(model_name=model_name)
    chunker = AgenticChunker(model_name=model_name)
    
    # Split text into paragraphs
    paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
    
    if max_paragraphs:
        paragraphs = paragraphs[:max_paragraphs]
    
    print(f"Processing {len(paragraphs)} paragraphs...")
    
    # Extract propositions from each paragraph
    all_propositions = []
    for i, para in enumerate(paragraphs):
        print(f"Processing paragraph {i+1}/{len(paragraphs)}...")
        propositions = extractor.get_propositions(para)
        all_propositions.extend(propositions)
    
    print(f"\n[green]Extracted {len(all_propositions)} propositions[/green]")
    
    # Use agentic chunker to group propositions
    print("\n[bold blue]#### Agentic Chunking ####[/bold blue]")
    chunker.add_propositions(all_propositions)
    
    # Get final chunks
    chunks = chunker.get_chunks(get_type='list_of_strings')
    
    print(f"\n[green]Created {len(chunks)} semantic chunks[/green]")
    chunker.pretty_print_chunks()
    
    return chunks


if __name__ == "__main__":
    # Example usage
    sample_text = """
    Artificial intelligence has made significant progress in recent years. Machine learning models can now perform complex tasks.
    
    Deep learning is a subset of machine learning. It uses neural networks with multiple layers. These networks can learn hierarchical representations.
    
    Natural language processing has benefited greatly from these advances. Models like GPT can generate human-like text. They can also understand context and nuance.
    """
    
    chunks = chunk_text_with_propositions(sample_text, max_paragraphs=3)
    
    print("\n[bold]Final Chunks:[/bold]")
    for i, chunk in enumerate(chunks):
        print(f"\n[yellow]Chunk {i+1}:[/yellow]")
        print(chunk)

### Parsers

In [ ]:
#CSV
"""
CSV and Excel parser using openpyxl and csv module.
Handles spreadsheet data extraction.
"""

from pathlib import Path
from typing import Dict, List
import csv
import openpyxl
from loguru import logger


class CSVParser:
    """Parser for CSV and Excel files."""
    
    def __init__(self):
        self.supported_extensions = ['.csv', '.xlsx', '.xls']
    
    def parse(self, file_path: Path) -> Dict[str, any]:
        """
        Parse a CSV or Excel file and extract data.
        
        Args:
            file_path: Path to the file
            
        Returns:
            Dictionary containing extracted content and metadata
        """
        logger.info(f"Parsing spreadsheet: {file_path}")
        
        try:
            if file_path.suffix.lower() == '.csv':
                return self._parse_csv(file_path)
            else:
                return self._parse_excel(file_path)
                
        except Exception as e:
            logger.error(f"Error parsing spreadsheet {file_path}: {e}")
            raise
    
    def _parse_csv(self, file_path: Path) -> Dict[str, any]:
        """Parse CSV file."""
        try:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                # Detect dialect
                sample = f.read(1024)
                f.seek(0)
                dialect = csv.Sniffer().sniff(sample)
                
                # Read CSV
                reader = csv.reader(f, dialect)
                data = list(reader)
            
            # Extract headers and rows
            headers = data[0] if data else []
            rows = data[1:] if len(data) > 1 else []
            
            # Format as text
            text_content = self._format_table_as_text(headers, rows)
            
            result = {
                'file_path': str(file_path),
                'file_name': file_path.name,
                'file_type': 'csv',
                'text': text_content,
                'data': {
                    'headers': headers,
                    'rows': rows
                },
                'metadata': {
                    'row_count': len(rows),
                    'column_count': len(headers)
                }
            }
            
            logger.success(f"Successfully parsed CSV: {file_path.name}")
            return result
            
        except Exception as e:
            logger.error(f"Error parsing CSV: {e}")
            raise
    
    def _parse_excel(self, file_path: Path) -> Dict[str, any]:
        """Parse Excel file using openpyxl."""
        try:
            workbook = openpyxl.load_workbook(file_path, data_only=True)
            sheets_data = []
            text_parts = []
            
            for sheet_name in workbook.sheetnames:
                sheet = workbook[sheet_name]
                
                # Extract data from sheet
                data = []
                for row in sheet.iter_rows(values_only=True):
                    # Convert None to empty string and all values to strings
                    row_data = [str(cell) if cell is not None else '' for cell in row]
                    data.append(row_data)
                
                if data:
                    headers = data[0] if data else []
                    rows = data[1:] if len(data) > 1 else []
                    
                    sheets_data.append({
                        'sheet_name': sheet_name,
                        'headers': headers,
                        'rows': rows,
                        'row_count': len(rows),
                        'column_count': len(headers)
                    })
                    
                    # Format sheet as text
                    text_parts.append(f"\n--- Sheet: {sheet_name} ---\n")
                    text_parts.append(self._format_table_as_text(headers, rows))
            
            workbook.close()
            
            result = {
                'file_path': str(file_path),
                'file_name': file_path.name,
                'file_type': 'excel',
                'text': '\n'.join(text_parts),
                'sheets': sheets_data,
                'metadata': {
                    'sheet_count': len(sheets_data),
                    'total_rows': sum(s['row_count'] for s in sheets_data)
                }
            }
            
            logger.success(f"Successfully parsed Excel: {file_path.name}")
            return result
            
        except Exception as e:
            logger.error(f"Error parsing Excel: {e}")
            raise
    
    def _format_table_as_text(self, headers: List[str], rows: List[List[str]], max_rows: int = 100) -> str:
        """Format table data as readable text."""
        text_parts = []
        
        # Add headers
        if headers:
            text_parts.append(' | '.join(headers))
            text_parts.append('-' * 80)
        
        # Add rows (limit to max_rows to avoid huge text blocks)
        for i, row in enumerate(rows[:max_rows]):
            text_parts.append(' | '.join(row))
        
        if len(rows) > max_rows:
            text_parts.append(f"\n... ({len(rows) - max_rows} more rows)")
        
        return '\n'.join(text_parts)
#dispatcher
"""
Document parser dispatcher.
Routes files to the appropriate parser based on file extension.
"""

from pathlib import Path
from typing import Dict, Optional
from loguru import logger

from parsers.pdf_parser import PDFParser
from parsers.docx_parser import DOCXParser
from parsers.pptx_parser import PPTXParser
from parsers.csv_parser import CSVParser


class ParserDispatcher:
    """
    Dispatcher that routes documents to the appropriate parser.
    """
    
    def __init__(self):
        """Initialize all parsers."""
        self.parsers = {
            'pdf': PDFParser(),
            'docx': DOCXParser(),
            'pptx': PPTXParser(),
            'csv': CSVParser()
        }
        
        # Build extension to parser mapping
        self.extension_map = {}
        for parser_name, parser in self.parsers.items():
            for ext in parser.supported_extensions:
                self.extension_map[ext.lower()] = parser_name
        
        logger.info(f"Initialized parser dispatcher with {len(self.parsers)} parsers")
        logger.debug(f"Supported extensions: {list(self.extension_map.keys())}")
    
    def parse(self, file_path: Path) -> Optional[Dict[str, any]]:
        """
        Parse a document by dispatching to the appropriate parser.
        
        Args:
            file_path: Path to the document file
            
        Returns:
            Dictionary containing parsed content and metadata, or None if unsupported
        """
        if not file_path.exists():
            logger.error(f"File not found: {file_path}")
            raise FileNotFoundError(f"File not found: {file_path}")
        
        # Get file extension
        extension = file_path.suffix.lower()
        
        # Find appropriate parser
        parser_name = self.extension_map.get(extension)
        
        if not parser_name:
            logger.warning(f"Unsupported file type: {extension} for file {file_path.name}")
            return None
        
        # Get parser and parse document
        parser = self.parsers[parser_name]
        logger.info(f"Dispatching {file_path.name} to {parser_name} parser")
        
        try:
            result = parser.parse(file_path)
            return result
        except Exception as e:
            logger.error(f"Failed to parse {file_path.name}: {e}")
            raise
    
    def is_supported(self, file_path: Path) -> bool:
        """
        Check if a file type is supported.
        
        Args:
            file_path: Path to check
            
        Returns:
            True if the file type is supported
        """
        extension = file_path.suffix.lower()
        return extension in self.extension_map
    
    def get_supported_extensions(self) -> list:
        """
        Get list of all supported file extensions.
        
        Returns:
            List of supported extensions
        """
        return list(self.extension_map.keys())


def parse_document(file_path: Path) -> Optional[Dict[str, any]]:
    """
    Convenience function to parse a document.
    
    Args:
        file_path: Path to the document
        
    Returns:
        Parsed document data or None if unsupported
    """
    dispatcher = ParserDispatcher()
    return dispatcher.parse(file_path)
#docx parser
"""
Word document parser using python-docx.
Handles text, tables, and document structure.
"""

from pathlib import Path
from typing import Dict, List
from docx import Document
from loguru import logger


class DOCXParser:
    """Parser for Microsoft Word documents."""
    
    def __init__(self):
        self.supported_extensions = ['.docx', '.doc']
    
    def parse(self, file_path: Path) -> Dict[str, any]:
        """
        Parse a Word document and extract text, tables, and metadata.
        
        Args:
            file_path: Path to the Word file
            
        Returns:
            Dictionary containing extracted content and metadata
        """
        logger.info(f"Parsing DOCX: {file_path}")
        
        try:
            doc = Document(file_path)
            
            # Extract text from paragraphs
            text_content = self._extract_text(doc)
            
            # Extract tables
            tables = self._extract_tables(doc)
            
            # Extract metadata
            metadata = self._extract_metadata(doc)
            
            result = {
                'file_path': str(file_path),
                'file_name': file_path.name,
                'file_type': 'docx',
                'text': text_content,
                'tables': tables,
                'metadata': metadata,
                'paragraph_count': len(doc.paragraphs),
                'table_count': len(doc.tables)
            }
            
            logger.success(f"Successfully parsed DOCX: {file_path.name}")
            return result
            
        except Exception as e:
            logger.error(f"Error parsing DOCX {file_path}: {e}")
            raise
    
    def _extract_text(self, doc: Document) -> str:
        """Extract text from all paragraphs."""
        text_parts = []
        
        for para in doc.paragraphs:
            if para.text.strip():
                # Preserve heading structure
                if para.style.name.startswith('Heading'):
                    level = para.style.name.replace('Heading ', '')
                    text_parts.append(f"\n{'#' * int(level) if level.isdigit() else '#'} {para.text}\n")
                else:
                    text_parts.append(para.text)
        
        return '\n'.join(text_parts)
    
    def _extract_tables(self, doc: Document) -> List[Dict]:
        """Extract tables from the document."""
        tables = []
        
        for table_idx, table in enumerate(doc.tables):
            try:
                # Extract table data
                table_data = []
                for row in table.rows:
                    row_data = [cell.text.strip() for cell in row.cells]
                    table_data.append(row_data)
                
                if table_data:
                    tables.append({
                        'table_index': table_idx,
                        'data': table_data,
                        'rows': len(table_data),
                        'columns': len(table_data[0]) if table_data else 0
                    })
            except Exception as e:
                logger.warning(f"Error extracting table {table_idx}: {e}")
                continue
        
        logger.info(f"Extracted {len(tables)} tables from DOCX")
        return tables
    
    def _extract_metadata(self, doc: Document) -> Dict:
        """Extract document metadata."""
        try:
            core_props = doc.core_properties
            
            return {
                'title': core_props.title or '',
                'author': core_props.author or '',
                'subject': core_props.subject or '',
                'keywords': core_props.keywords or '',
                'comments': core_props.comments or '',
                'created': str(core_props.created) if core_props.created else '',
                'modified': str(core_props.modified) if core_props.modified else '',
                'last_modified_by': core_props.last_modified_by or ''
            }
        except Exception as e:
            logger.warning(f"Error extracting metadata: {e}")
            return {}
#PDF
"""
PDF document parser using pdfplumber and pymupdf.
Handles text extraction, tables, and multi-column layouts.
"""

from pathlib import Path
from typing import Dict, List, Optional
import pdfplumber
import fitz  # pymupdf
from loguru import logger


class PDFParser:
    """Parser for PDF documents with table and layout support."""
    
    def __init__(self):
        self.supported_extensions = ['.pdf']
    
    def parse(self, file_path: Path) -> Dict[str, any]:
        """
        Parse a PDF file and extract text, tables, and metadata.
        
        Args:
            file_path: Path to the PDF file
            
        Returns:
            Dictionary containing extracted content and metadata
        """
        logger.info(f"Parsing PDF: {file_path}")
        
        try:
            # Extract text and tables using pdfplumber
            text_content = self._extract_text_pdfplumber(file_path)
            tables = self._extract_tables(file_path)
            
            # Extract metadata using pymupdf
            metadata = self._extract_metadata(file_path)
            
            result = {
                'file_path': str(file_path),
                'file_name': file_path.name,
                'file_type': 'pdf',
                'text': text_content,
                'tables': tables,
                'metadata': metadata,
                'page_count': metadata.get('page_count', 0)
            }
            
            logger.success(f"Successfully parsed PDF: {file_path.name}")
            return result
            
        except Exception as e:
            logger.error(f"Error parsing PDF {file_path}: {e}")
            raise
    
    def _extract_text_pdfplumber(self, file_path: Path) -> str:
        """Extract text using pdfplumber with layout preservation."""
        text_parts = []
        
        with pdfplumber.open(file_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                try:
                    # Extract text with layout
                    page_text = page.extract_text(layout=True)
                    if page_text:
                        text_parts.append(f"\n--- Page {page_num} ---\n")
                        text_parts.append(page_text)
                except Exception as e:
                    logger.warning(f"Error extracting text from page {page_num}: {e}")
                    continue
        
        return '\n'.join(text_parts)
    
    def _extract_tables(self, file_path: Path) -> List[Dict]:
        """Extract tables from PDF."""
        tables = []
        
        with pdfplumber.open(file_path) as pdf:
            for page_num, page in enumerate(pdf.pages, 1):
                try:
                    page_tables = page.extract_tables()
                    for table_idx, table in enumerate(page_tables):
                        if table:
                            tables.append({
                                'page': page_num,
                                'table_index': table_idx,
                                'data': table,
                                'rows': len(table),
                                'columns': len(table[0]) if table else 0
                            })
                except Exception as e:
                    logger.warning(f"Error extracting tables from page {page_num}: {e}")
                    continue
        
        logger.info(f"Extracted {len(tables)} tables from PDF")
        return tables
    
    def _extract_metadata(self, file_path: Path) -> Dict:
        """Extract metadata using pymupdf."""
        try:
            doc = fitz.open(file_path)
            metadata = doc.metadata or {}
            
            result = {
                'title': metadata.get('title', ''),
                'author': metadata.get('author', ''),
                'subject': metadata.get('subject', ''),
                'keywords': metadata.get('keywords', ''),
                'creator': metadata.get('creator', ''),
                'producer': metadata.get('producer', ''),
                'creation_date': metadata.get('creationDate', ''),
                'modification_date': metadata.get('modDate', ''),
                'page_count': doc.page_count
            }
            
            doc.close()
            return result
            
        except Exception as e:
            logger.warning(f"Error extracting metadata: {e}")
            return {'page_count': 0}
#pptx
"""
PowerPoint parser using python-pptx.
Extracts text from slides, notes, and tables.
"""

from pathlib import Path
from typing import Dict, List
from pptx import Presentation
from loguru import logger


class PPTXParser:
    """Parser for Microsoft PowerPoint presentations."""
    
    def __init__(self):
        self.supported_extensions = ['.pptx', '.ppt']
    
    def parse(self, file_path: Path) -> Dict[str, any]:
        """
        Parse a PowerPoint file and extract text and metadata.
        
        Args:
            file_path: Path to the PowerPoint file
            
        Returns:
            Dictionary containing extracted content and metadata
        """
        logger.info(f"Parsing PPTX: {file_path}")
        
        try:
            prs = Presentation(file_path)
            
            # Extract text from slides
            slides_content = self._extract_slides(prs)
            
            # Combine all text
            text_content = self._format_slides_text(slides_content)
            
            # Extract metadata
            metadata = self._extract_metadata(prs)
            
            result = {
                'file_path': str(file_path),
                'file_name': file_path.name,
                'file_type': 'pptx',
                'text': text_content,
                'slides': slides_content,
                'metadata': metadata,
                'slide_count': len(prs.slides)
            }
            
            logger.success(f"Successfully parsed PPTX: {file_path.name}")
            return result
            
        except Exception as e:
            logger.error(f"Error parsing PPTX {file_path}: {e}")
            raise
    
    def _extract_slides(self, prs: Presentation) -> List[Dict]:
        """Extract content from all slides."""
        slides_content = []
        
        for slide_idx, slide in enumerate(prs.slides, 1):
            try:
                slide_text = []
                
                # Extract text from all shapes
                for shape in slide.shapes:
                    if hasattr(shape, "text") and shape.text.strip():
                        slide_text.append(shape.text.strip())
                    
                    # Extract text from tables
                    if shape.has_table:
                        table_text = self._extract_table_from_shape(shape)
                        if table_text:
                            slide_text.append(table_text)
                
                # Extract notes
                notes_text = ""
                if slide.has_notes_slide:
                    notes_slide = slide.notes_slide
                    if notes_slide.notes_text_frame:
                        notes_text = notes_slide.notes_text_frame.text.strip()
                
                slides_content.append({
                    'slide_number': slide_idx,
                    'content': '\n'.join(slide_text),
                    'notes': notes_text
                })
                
            except Exception as e:
                logger.warning(f"Error extracting slide {slide_idx}: {e}")
                continue
        
        logger.info(f"Extracted content from {len(slides_content)} slides")
        return slides_content
    
    def _extract_table_from_shape(self, shape) -> str:
        """Extract text from a table shape."""
        try:
            table = shape.table
            table_text = []
            
            for row in table.rows:
                row_text = [cell.text.strip() for cell in row.cells]
                table_text.append(' | '.join(row_text))
            
            return '\n'.join(table_text)
        except Exception as e:
            logger.warning(f"Error extracting table: {e}")
            return ""
    
    def _format_slides_text(self, slides_content: List[Dict]) -> str:
        """Format slides content into a single text string."""
        text_parts = []
        
        for slide in slides_content:
            text_parts.append(f"\n--- Slide {slide['slide_number']} ---\n")
            text_parts.append(slide['content'])
            
            if slide['notes']:
                text_parts.append(f"\nNotes: {slide['notes']}")
        
        return '\n'.join(text_parts)
    
    def _extract_metadata(self, prs: Presentation) -> Dict:
        """Extract presentation metadata."""
        try:
            core_props = prs.core_properties
            
            return {
                'title': core_props.title or '',
                'author': core_props.author or '',
                'subject': core_props.subject or '',
                'keywords': core_props.keywords or '',
                'comments': core_props.comments or '',
                'created': str(core_props.created) if core_props.created else '',
                'modified': str(core_props.modified) if core_props.modified else '',
                'last_modified_by': core_props.last_modified_by or '',
                'revision': core_props.revision
            }
        except Exception as e:
            logger.warning(f"Error extracting metadata: {e}")
            return {}


In [ ]:
#BM25 Store
"""
BM25 sparse index management for keyword-based retrieval.
"""

from typing import List, Dict, Optional
import pickle
from pathlib import Path
from rank_bm25 import BM25Okapi
from loguru import logger

from config.settings import DATA_DIR


class BM25Store:
    """Manages BM25 index for sparse keyword retrieval."""
    
    def __init__(self, index_name: str = "bm25_index"):
        """
        Initialize BM25 store.
        
        Args:
            index_name: Name for the index file
        """
        self.index_name = index_name
        self.index_path = DATA_DIR / f"{index_name}.pkl"
        self.metadata_path = DATA_DIR / f"{index_name}_metadata.pkl"
        
        self.bm25 = None
        self.corpus = []
        self.tokenized_corpus = []
        self.chunk_metadata = []
        
        # Try to load existing index
        self.load_index()
        
        logger.info(f"Initialized BM25 store: {index_name}")
    
    def _tokenize(self, text: str) -> List[str]:
        """
        Simple tokenization for BM25.
        
        Args:
            text: Text to tokenize
            
        Returns:
            List of tokens
        """
        # Simple whitespace tokenization with lowercasing
        # You can enhance this with proper NLP tokenization if needed
        return text.lower().split()
    
    def add_chunks(self, chunks: List[Dict]) -> int:
        """
        Add chunks to the BM25 index.
        
        Args:
            chunks: List of chunk dictionaries with 'text' and 'metadata'
            
        Returns:
            Number of chunks added
        """
        if not chunks:
            logger.warning("No chunks provided to add")
            return 0
        
        logger.info(f"Adding {len(chunks)} chunks to BM25 index")
        
        for chunk in chunks:
            text = chunk.get('text', '')
            if not text.strip():
                continue
            
            # Tokenize and add to corpus
            tokens = self._tokenize(text)
            self.tokenized_corpus.append(tokens)
            self.corpus.append(text)
            self.chunk_metadata.append({
                'chunk_id': chunk.get('chunk_id', f"chunk_{len(self.corpus)}"),
                'metadata': chunk.get('metadata', {})
            })
        
        # Rebuild BM25 index
        if self.tokenized_corpus:
            self.bm25 = BM25Okapi(self.tokenized_corpus)
            logger.success(f"Built BM25 index with {len(self.corpus)} documents")
        
        return len(chunks)
    
    def search(
        self,
        query: str,
        top_k: int = 10,
        filter_metadata: Optional[Dict] = None
    ) -> List[Dict]:
        """
        Search using BM25 keyword matching.
        
        Args:
            query: Search query
            top_k: Number of results to return
            filter_metadata: Optional metadata filters (not implemented for BM25)
            
        Returns:
            List of search results with text, metadata, and scores
        """
        if not query or not query.strip():
            logger.warning("Empty query provided")
            return []
        
        if not self.bm25:
            logger.warning("BM25 index is empty")
            return []
        
        try:
            # Tokenize query
            query_tokens = self._tokenize(query)
            
            # Get BM25 scores
            scores = self.bm25.get_scores(query_tokens)
            
            # Get top-k indices
            top_indices = sorted(
                range(len(scores)),
                key=lambda i: scores[i],
                reverse=True
            )[:top_k]
            
            # Format results
            results = []
            for idx in top_indices:
                if scores[idx] > 0:  # Only include results with positive scores
                    results.append({
                        'chunk_id': self.chunk_metadata[idx]['chunk_id'],
                        'text': self.corpus[idx],
                        'metadata': self.chunk_metadata[idx]['metadata'],
                        'score': float(scores[idx]),
                        'rank': len(results) + 1
                    })
            
            logger.info(f"Found {len(results)} BM25 results for query")
            return results
            
        except Exception as e:
            logger.error(f"BM25 search error: {e}")
            return []
    
    def save_index(self):
        """Save BM25 index and metadata to disk."""
        try:
            # Save BM25 index
            with open(self.index_path, 'wb') as f:
                pickle.dump({
                    'bm25': self.bm25,
                    'corpus': self.corpus,
                    'tokenized_corpus': self.tokenized_corpus
                }, f)
            
            # Save metadata
            with open(self.metadata_path, 'wb') as f:
                pickle.dump(self.chunk_metadata, f)
            
            logger.success(f"Saved BM25 index to {self.index_path}")
            
        except Exception as e:
            logger.error(f"Error saving BM25 index: {e}")
    
    def load_index(self) -> bool:
        """
        Load BM25 index from disk.
        
        Returns:
            True if loaded successfully
        """
        if not self.index_path.exists():
            logger.info("No existing BM25 index found")
            return False
        
        try:
            # Load BM25 index
            with open(self.index_path, 'rb') as f:
                data = pickle.load(f)
                self.bm25 = data['bm25']
                self.corpus = data['corpus']
                self.tokenized_corpus = data['tokenized_corpus']
            
            # Load metadata
            if self.metadata_path.exists():
                with open(self.metadata_path, 'rb') as f:
                    self.chunk_metadata = pickle.load(f)
            
            logger.success(f"Loaded BM25 index with {len(self.corpus)} documents")
            return True
            
        except Exception as e:
            logger.error(f"Error loading BM25 index: {e}")
            return False
    
    def reset_index(self):
        """Reset the BM25 index."""
        self.bm25 = None
        self.corpus = []
        self.tokenized_corpus = []
        self.chunk_metadata = []
        
        # Delete index files
        if self.index_path.exists():
            self.index_path.unlink()
        if self.metadata_path.exists():
            self.metadata_path.unlink()
        
        logger.success("Reset BM25 index")
    
    def get_stats(self) -> Dict:
        """Get index statistics."""
        return {
            'index_name': self.index_name,
            'document_count': len(self.corpus),
            'index_path': str(self.index_path),
            'index_exists': self.bm25 is not None
        }


def create_bm25_store(index_name: str = "bm25_index") -> BM25Store:
    """
    Convenience function to create a BM25 store.
    
    Args:
        index_name: Name for the index
        
    Returns:
        BM25Store instance
    """
    return BM25Store(index_name=index_name)


In [ ]:
#Chunker
"""
Section-aware and semantic chunking for RAG pipeline.
Integrates with the agentic chunker for intelligent document segmentation.
"""

from pathlib import Path
from typing import List, Dict, Optional
from loguru import logger
import sys

# Add ingest directory to path to import agentic chunker
sys.path.insert(0, str(Path(__file__).parent.parent))

from ingest.agentic_chunker import AgenticChunker
from ingest.chunker import PropositionExtractor
from config.settings import CHUNK_SIZE, CHUNK_OVERLAP, OLLAMA_MODEL


class DocumentChunker:
    """
    Handles document chunking with multiple strategies:
    1. Section-aware chunking (preserves document structure)
    2. Semantic chunking (uses agentic chunker for intelligent grouping)
    3. Fixed-size chunking (fallback for simple cases)
    """
    
    def __init__(
        self,
        chunk_size: int = CHUNK_SIZE,
        chunk_overlap: int = CHUNK_OVERLAP,
        model_name: str = OLLAMA_MODEL,
        use_agentic: bool = True
    ):
        """
        Initialize the chunker.
        
        Args:
            chunk_size: Maximum size of chunks in characters
            chunk_overlap: Overlap between chunks
            model_name: Ollama model for agentic chunking
            use_agentic: Whether to use agentic chunking (slower but smarter)
        """
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.model_name = model_name
        self.use_agentic = use_agentic
        
        if self.use_agentic:
            self.proposition_extractor = PropositionExtractor(model_name=model_name)
            self.agentic_chunker = AgenticChunker(model_name=model_name)
            logger.info(f"Initialized agentic chunker with model: {model_name}")
        else:
            logger.info("Using fixed-size chunking")
    
    def chunk_document(
        self,
        text: str,
        metadata: Optional[Dict] = None,
        strategy: str = "auto"
    ) -> List[Dict]:
        """
        Chunk a document using the specified strategy.
        
        Args:
            text: Document text to chunk
            metadata: Optional metadata to attach to chunks
            strategy: Chunking strategy ("auto", "agentic", "section", "fixed")
            
        Returns:
            List of chunk dictionaries with text and metadata
        """
        if not text or not text.strip():
            logger.warning("Empty text provided for chunking")
            return []
        
        metadata = metadata or {}
        
        # Choose strategy
        if strategy == "auto":
            # Use agentic for shorter documents, fixed for very long ones
            if len(text) < 50000 and self.use_agentic:
                strategy = "agentic"
            elif self._has_clear_sections(text):
                strategy = "section"
            else:
                strategy = "fixed"
        
        logger.info(f"Chunking document using strategy: {strategy}")
        
        # Apply chunking strategy
        if strategy == "agentic" and self.use_agentic:
            chunks = self._chunk_agentic(text, metadata)
        elif strategy == "section":
            chunks = self._chunk_by_section(text, metadata)
        else:
            chunks = self._chunk_fixed_size(text, metadata)
        
        logger.success(f"Created {len(chunks)} chunks")
        return chunks
    
    def _chunk_agentic(self, text: str, metadata: Dict) -> List[Dict]:
        """Use agentic chunker for semantic grouping."""
        try:
            # Split into paragraphs
            paragraphs = [p.strip() for p in text.split("\n\n") if p.strip()]
            
            # Extract propositions
            all_propositions = []
            for para in paragraphs:
                propositions = self.proposition_extractor.get_propositions(para)
                all_propositions.extend(propositions)
            
            logger.info(f"Extracted {len(all_propositions)} propositions")
            
            # Use agentic chunker to group
            self.agentic_chunker.add_propositions(all_propositions)
            
            # Get chunks
            chunk_dict = self.agentic_chunker.get_chunks(get_type='dict')
            
            # Format as list with metadata
            chunks = []
            for chunk_id, chunk_data in chunk_dict.items():
                chunk_text = " ".join(chunk_data['propositions'])
                chunks.append({
                    'text': chunk_text,
                    'chunk_id': chunk_id,
                    'title': chunk_data.get('title', ''),
                    'summary': chunk_data.get('summary', ''),
                    'chunk_index': chunk_data.get('chunk_index', 0),
                    'metadata': {
                        **metadata,
                        'chunking_strategy': 'agentic',
                        'proposition_count': len(chunk_data['propositions'])
                    }
                })
            
            # Reset chunker for next document
            self.agentic_chunker.chunks = {}
            
            return chunks
            
        except Exception as e:
            logger.error(f"Agentic chunking failed: {e}, falling back to fixed-size")
            return self._chunk_fixed_size(text, metadata)
    
    def _chunk_by_section(self, text: str, metadata: Dict) -> List[Dict]:
        """Chunk by document sections (headers, paragraphs)."""
        chunks = []
        
        # Split by common section markers
        sections = self._split_into_sections(text)
        
        for i, section in enumerate(sections):
            if not section.strip():
                continue
            
            # If section is too large, split it further
            if len(section) > self.chunk_size:
                sub_chunks = self._chunk_fixed_size(section, metadata)
                for j, sub_chunk in enumerate(sub_chunks):
                    sub_chunk['metadata']['section_index'] = i
                    sub_chunk['metadata']['sub_chunk_index'] = j
                    chunks.append(sub_chunk)
            else:
                chunks.append({
                    'text': section.strip(),
                    'chunk_id': f"section_{i}",
                    'chunk_index': i,
                    'metadata': {
                        **metadata,
                        'chunking_strategy': 'section',
                        'section_index': i
                    }
                })
        
        return chunks
    
    def _chunk_fixed_size(self, text: str, metadata: Dict) -> List[Dict]:
        """Simple fixed-size chunking with overlap."""
        chunks = []
        start = 0
        chunk_index = 0
        
        while start < len(text):
            # Get chunk
            end = start + self.chunk_size
            chunk_text = text[start:end]
            
            # Try to break at sentence boundary
            if end < len(text):
                last_period = chunk_text.rfind('.')
                last_newline = chunk_text.rfind('\n')
                break_point = max(last_period, last_newline)
                
                if break_point > self.chunk_size * 0.5:  # At least 50% of chunk size
                    chunk_text = chunk_text[:break_point + 1]
                    end = start + break_point + 1
            
            chunks.append({
                'text': chunk_text.strip(),
                'chunk_id': f"chunk_{chunk_index}",
                'chunk_index': chunk_index,
                'metadata': {
                    **metadata,
                    'chunking_strategy': 'fixed',
                    'start_char': start,
                    'end_char': end
                }
            })
            
            # Move to next chunk with overlap
            start = end - self.chunk_overlap
            chunk_index += 1
        
        return chunks
    
    def _has_clear_sections(self, text: str) -> bool:
        """Check if text has clear section markers."""
        # Look for markdown headers or numbered sections
        lines = text.split('\n')
        header_count = sum(1 for line in lines if line.strip().startswith('#') or 
                          (line.strip() and line.strip()[0].isdigit() and '. ' in line[:5]))
        
        return header_count > 3
    
    def _split_into_sections(self, text: str) -> List[str]:
        """Split text into sections based on headers."""
        sections = []
        current_section = []
        
        for line in text.split('\n'):
            # Check if line is a header
            is_header = (
                line.strip().startswith('#') or
                (line.strip() and line.strip()[0].isdigit() and '. ' in line[:5]) or
                (line.strip() and line.strip().isupper() and len(line.strip()) < 100)
            )
            
            if is_header and current_section:
                # Save previous section
                sections.append('\n'.join(current_section))
                current_section = [line]
            else:
                current_section.append(line)
        
        # Add last section
        if current_section:
            sections.append('\n'.join(current_section))
        
        return sections


def chunk_document(
    text: str,
    metadata: Optional[Dict] = None,
    strategy: str = "auto",
    use_agentic: bool = True
) -> List[Dict]:
    """
    Convenience function to chunk a document.
    
    Args:
        text: Document text
        metadata: Optional metadata
        strategy: Chunking strategy
        use_agentic: Whether to use agentic chunking
        
    Returns:
        List of chunks
    """
    chunker = DocumentChunker(use_agentic=use_agentic)
    return chunker.chunk_document(text, metadata, strategy)


In [ ]:
#Embedder
"""
Embedding wrapper using Ollama for local embeddings.
"""

from typing import List, Optional
from loguru import logger
import requests
from config.settings import OLLAMA_BASE_URL, OLLAMA_EMBEDDING_MODEL


class OllamaEmbedder:
    """Wrapper for Ollama embedding generation."""
    
    def __init__(
        self,
        model: str = OLLAMA_EMBEDDING_MODEL,
        base_url: str = OLLAMA_BASE_URL
    ):
        """
        Initialize the embedder.
        
        Args:
            model: Ollama embedding model name
            base_url: Ollama server URL
        """
        self.model = model
        self.base_url = base_url.rstrip('/')
        self.embed_url = f"{self.base_url}/api/embeddings"
        
        logger.info(f"Initialized Ollama embedder with model: {model}")
        self._verify_connection()
    
    def _verify_connection(self):
        """Verify connection to Ollama server."""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=5)
            if response.status_code == 200:
                logger.success("Connected to Ollama server")
            else:
                logger.warning(f"Ollama server returned status {response.status_code}")
        except Exception as e:
            logger.error(f"Failed to connect to Ollama server: {e}")
            logger.warning("Make sure Ollama is running: ollama serve")
    
    def embed_text(self, text: str) -> Optional[List[float]]:
        """
        Generate embedding for a single text.
        
        Args:
            text: Text to embed
            
        Returns:
            Embedding vector or None if failed
        """
        if not text or not text.strip():
            logger.warning("Empty text provided for embedding")
            return None
        
        try:
            response = requests.post(
                self.embed_url,
                json={
                    "model": self.model,
                    "prompt": text
                },
                timeout=30
            )
            
            if response.status_code == 200:
                data = response.json()
                embedding = data.get('embedding')
                
                if embedding:
                    return embedding
                else:
                    logger.error("No embedding in response")
                    return None
            else:
                logger.error(f"Embedding request failed: {response.status_code}")
                return None
                
        except Exception as e:
            logger.error(f"Error generating embedding: {e}")
            return None
    
    def embed_batch(self, texts: List[str], show_progress: bool = True) -> List[Optional[List[float]]]:
        """
        Generate embeddings for multiple texts.
        
        Args:
            texts: List of texts to embed
            show_progress: Whether to log progress
            
        Returns:
            List of embedding vectors
        """
        embeddings = []
        total = len(texts)
        
        for i, text in enumerate(texts):
            if show_progress and (i + 1) % 10 == 0:
                logger.info(f"Embedding progress: {i + 1}/{total}")
            
            embedding = self.embed_text(text)
            embeddings.append(embedding)
        
        if show_progress:
            successful = sum(1 for e in embeddings if e is not None)
            logger.success(f"Generated {successful}/{total} embeddings")
        
        return embeddings
    
    def get_embedding_dimension(self) -> Optional[int]:
        """
        Get the dimension of embeddings from this model.
        
        Returns:
            Embedding dimension or None if failed
        """
        test_embedding = self.embed_text("test")
        if test_embedding:
            return len(test_embedding)
        return None


def embed_text(text: str, model: str = OLLAMA_EMBEDDING_MODEL) -> Optional[List[float]]:
    """
    Convenience function to embed a single text.
    
    Args:
        text: Text to embed
        model: Embedding model name
        
    Returns:
        Embedding vector
    """
    embedder = OllamaEmbedder(model=model)
    return embedder.embed_text(text)


def embed_batch(texts: List[str], model: str = OLLAMA_EMBEDDING_MODEL) -> List[Optional[List[float]]]:
    """
    Convenience function to embed multiple texts.
    
    Args:
        texts: List of texts
        model: Embedding model name
        
    Returns:
        List of embedding vectors
    """
    embedder = OllamaEmbedder(model=model)
    return embedder.embed_batch(texts)


In [ ]:
#VECTOR STORE
"""
ChromaDB vector store integration for semantic search.
"""

from typing import List, Dict, Optional
from pathlib import Path
import chromadb
from chromadb.config import Settings
from loguru import logger

from config.settings import CHROMA_PERSIST_DIR, CHROMA_COLLECTION_NAME
from embedder import HuggingFaceEmbedder


class ChromaVectorStore:
    """Manages ChromaDB vector store for document retrieval."""
    
    def __init__(
        self,
        collection_name: str = CHROMA_COLLECTION_NAME,
        persist_directory: str = CHROMA_PERSIST_DIR,
        embedder: Optional[HuggingFaceEmbedder] = None
    ):
        """
        Initialize ChromaDB vector store.
        
        Args:
            collection_name: Name of the collection
            persist_directory: Directory for persistent storage
            embedder: Optional embedder instance (HuggingFaceEmbedder)
        """
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.embedder = embedder or HuggingFaceEmbedder()
        
        # Initialize ChromaDB client
        self.client = chromadb.PersistentClient(
            path=persist_directory,
            settings=Settings(
                anonymized_telemetry=False,
                allow_reset=True
            )
        )
        
        # Get or create collection
        self.collection = self.client.get_or_create_collection(
            name=collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        
        logger.info(f"Initialized ChromaDB collection: {collection_name}")
        logger.info(f"Persist directory: {persist_directory}")
        logger.info(f"Current document count: {self.collection.count()}")
    
    def add_chunks(
        self,
        chunks: List[Dict],
        batch_size: int = 100
    ) -> int:
        """
        Add document chunks to the vector store.
        
        Args:
            chunks: List of chunk dictionaries with 'text' and 'metadata'
            batch_size: Number of chunks to process at once
            
        Returns:
            Number of chunks added
        """
        if not chunks:
            logger.warning("No chunks provided to add")
            return 0
        
        logger.info(f"Adding {len(chunks)} chunks to vector store")
        
        added_count = 0
        
        # Process in batches
        for i in range(0, len(chunks), batch_size):
            batch = chunks[i:i + batch_size]
            
            try:
                # Extract data
                texts = [chunk['text'] for chunk in batch]
                ids = [chunk.get('chunk_id', f"chunk_{i + j}") for j, chunk in enumerate(batch)]
                metadatas = [chunk.get('metadata', {}) for chunk in batch]
                
                # Generate embeddings
                embeddings = self.embedder.embed_batch(texts, show_progress=False)
                
                # Filter out failed embeddings
                valid_data = [
                    (id_, text, emb, meta)
                    for id_, text, emb, meta in zip(ids, texts, embeddings, metadatas)
                    if emb is not None
                ]
                
                if not valid_data:
                    logger.warning(f"No valid embeddings in batch {i // batch_size + 1}")
                    continue
                
                valid_ids, valid_texts, valid_embeddings, valid_metadatas = zip(*valid_data)
                
                # Add to collection
                self.collection.add(
                    ids=list(valid_ids),
                    documents=list(valid_texts),
                    embeddings=list(valid_embeddings),
                    metadatas=list(valid_metadatas)
                )
                
                added_count += len(valid_data)
                logger.info(f"Added batch {i // batch_size + 1}: {len(valid_data)} chunks")
                
            except Exception as e:
                logger.error(f"Error adding batch {i // batch_size + 1}: {e}")
                continue
        
        logger.success(f"Successfully added {added_count}/{len(chunks)} chunks")
        return added_count
    
    def search(
        self,
        query: str,
        top_k: int = 10,
        filter_metadata: Optional[Dict] = None
    ) -> List[Dict]:
        """
        Search for similar chunks using semantic similarity.
        
        Args:
            query: Search query
            top_k: Number of results to return
            filter_metadata: Optional metadata filters
            
        Returns:
            List of search results with text, metadata, and scores
        """
        if not query or not query.strip():
            logger.warning("Empty query provided")
            return []
        
        try:
            # Generate query embedding
            query_embedding = self.embedder.embed_text(query)
            
            if not query_embedding:
                logger.error("Failed to generate query embedding")
                return []
            
            # Search
            results = self.collection.query(
                query_embeddings=[query_embedding],
                n_results=top_k,
                where=filter_metadata
            )
            
            # Format results
            formatted_results = []
            
            if results['ids'] and len(results['ids']) > 0:
                for i in range(len(results['ids'][0])):
                    formatted_results.append({
                        'chunk_id': results['ids'][0][i],
                        'text': results['documents'][0][i],
                        'metadata': results['metadatas'][0][i] if results['metadatas'] else {},
                        'distance': results['distances'][0][i] if results['distances'] else None,
                        'score': 1 - results['distances'][0][i] if results['distances'] else None
                    })
            
            logger.info(f"Found {len(formatted_results)} results for query")
            return formatted_results
            
        except Exception as e:
            logger.error(f"Search error: {e}")
            return []
    
    def delete_collection(self):
        """Delete the entire collection."""
        try:
            self.client.delete_collection(name=self.collection_name)
            logger.success(f"Deleted collection: {self.collection_name}")
        except Exception as e:
            logger.error(f"Error deleting collection: {e}")
    
    def reset_collection(self):
        """Reset the collection (delete and recreate)."""
        self.delete_collection()
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"hnsw:space": "cosine"}
        )
        logger.success(f"Reset collection: {self.collection_name}")
    
    def get_stats(self) -> Dict:
        """Get collection statistics."""
        return {
            'collection_name': self.collection_name,
            'document_count': self.collection.count(),
            'persist_directory': self.persist_directory
        }


def create_vector_store(collection_name: str = CHROMA_COLLECTION_NAME) -> ChromaVectorStore:
    """
    Convenience function to create a vector store.
    
    Args:
        collection_name: Name of the collection
        
    Returns:
        ChromaVectorStore instance
    """
    return ChromaVectorStore(collection_name=collection_name)
